# Astra ADAS M5 - Kaggle Training and Evaluation

This Kaggle notebook trains a separate YOLOv8s candidate on IDD-D, compares it with an optional existing checkpoint, and exports outputs under `/kaggle/working` without replacing your local model.

## Before running

1. In Kaggle, create a notebook and choose **Add Input**. Search for and attach [Indian Driving Dataset-Detections (YOLOv11)](https://www.kaggle.com/datasets/redzapdos123/indian-driving-dataset-detections-yolov11). Its public listing documents 15 classes and 33,569 train / 4,196 validation / 4,197 test images, matching the local dataset's documented schema and split counts. Kaggle mounts attached data read-only under `/kaggle/input`; the setup cell searches for the `train/images` folder, including nested dataset-folder layouts.
2. For a paired baseline comparison, also use **Add Input** to attach a Kaggle Dataset containing the current `best.pt`. The setup cell auto-detects one `best.pt`; if there are multiple, set the path explicitly there.
3. Set **Accelerator = GPU** in notebook settings. Kaggle runs this notebook on its hosted GPU; your laptop's RTX 4050 is not used by a Kaggle session. Enable **Internet** if Kaggle needs to install packages or download the YOLOv8s pretrained weights. Internet availability varies by account/settings; you can instead attach `yolov8s.pt` as an input and set `PRETRAINED_WEIGHTS` to its path in the training cell.
4. Run cells from top to bottom. Training outputs are stored in `/kaggle/working/astra_adas_training`; use **Save Version** with notebook output enabled to preserve/download them. Kaggle input datasets cannot be modified.

The Kaggle listing is approximately 22 GB. The matching title, class list, and split counts strongly indicate this is the same dataset family/version used locally; they do not prove every file is byte-identical. Kaggle inputs are read-only. The notebook creates a writable dataset view using symlinks to the original images and copied labels: malformed annotation rows are omitted from training labels, while any validation/test image with a malformed annotation is excluded from that evaluation split. The original source is unchanged, and a JSON audit lists every omitted row/image. Review that report: metrics use the reduced clean validation/test sets and are not directly comparable to reports using all original images. For formal evaluation, correct labels at the source and rerun. The dataset is licensed CC BY-NC-SA 4.0; retain attribution and follow the non-commercial/share-alike terms.

## Project status and evaluation

- **Prototype path exists:** video-file input, YOLO 15-class detection, ByteTrack IDs, lane/hazard heuristics, relative-depth/motion/TTC telemetry, traffic-light HSV estimates, sign counts, and Streamlit annotated-video/CSV outputs.
- **Detection improvement demonstrated (not retraining):** on the same evenly sampled 200-image IDD-D validation subset, the existing checkpoint's mAP@50 was 0.5045 at 640px and 0.5602 at 960px; Indian-specific mAP@50 was 0.4870 and 0.5194. The 960px CPU path was slower (15.3 vs 29.2 FPS). These are subset validation results, not held-out test results.
- **Still partial or missing:** live camera/CARLA input, signal-to-ego-lane association and labeled signal-state evaluation, traffic-sign type recognition, hazard subclasses, calibrated metric depth/speed, tracking ground-truth metrics, validated driving policy/audio alerts, and dashboard feedback/retraining.
- This notebook compares mAP (the standard object-detection quality metric) and batch-one validation FPS at 640px, 768px, and 960px. Validation FPS is detector-only on the Kaggle GPU, not end-to-end dashboard FPS.
- Use train for fitting and validation for comparison/model selection. Run test only once the candidate is frozen; do not tune from test results.

There is no meaningful guarantee of 100% detection accuracy: image labels can be ambiguous, classes are imbalanced, and unseen scenes differ from the training set. This notebook aims to improve measured validation mAP and reports held-out test performance only after model selection. It prints and saves per-class training counts so rare classes can be identified; more correctly labeled examples are usually needed to improve those classes reliably. Training improves only object detection; it does not complete the other ADAS blocks or establish safety. See the project README for the full implementation checklist and limitations.

In [ ]:
%pip -q install "ultralytics>=8.2.0" pyyaml

from pathlib import Path
from collections import Counter
from datetime import datetime
import json
import os
import shutil
import zipfile
import yaml
import torch
from ultralytics import YOLO

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", gpu.name)
    print(f"GPU memory: {gpu.total_memory / (1024 ** 3):.1f} GiB")
else:
    print("Enable a Kaggle GPU accelerator before training; this notebook will stop before training if CUDA is unavailable.")


In [ ]:
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
PROJECT_DIR = WORK_ROOT / 'astra_adas_training'
RUNS_DIR = PROJECT_DIR / 'runs'
EXPORT_DIR = PROJECT_DIR / 'exports'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# If Kaggle attaches multiple datasets containing train/images, set this to the
# exact dataset folder under /kaggle/input. Otherwise leave as None for discovery.
DATASET_ROOT_OVERRIDE = None
if DATASET_ROOT_OVERRIDE is not None:
    DATA_DIR = Path(DATASET_ROOT_OVERRIDE)
else:
    dataset_roots = sorted({p.parent.parent for p in INPUT_ROOT.rglob('train/images') if p.is_dir()})
    if len(dataset_roots) != 1:
        raise RuntimeError(
            f'Expected exactly one attached dataset containing train/images, found {len(dataset_roots)}: '
            f'{dataset_roots}. Set DATASET_ROOT_OVERRIDE to the dataset folder.'
        )
    DATA_DIR = dataset_roots[0]

# Optional baseline comparison. If no best.pt is attached, comparison is skipped.
# If more than one exists, set BASELINE_OVERRIDE to the desired exact file path.
BASELINE_OVERRIDE = None
if BASELINE_OVERRIDE is not None:
    BASELINE_WEIGHTS = Path(BASELINE_OVERRIDE)
else:
    baseline_candidates = sorted(INPUT_ROOT.rglob('best.pt'))
    if len(baseline_candidates) == 1:
        BASELINE_WEIGHTS = baseline_candidates[0]
    elif len(baseline_candidates) == 0:
        BASELINE_WEIGHTS = None
        print('No best.pt found in Kaggle inputs; baseline comparison will be skipped.')
    else:
        BASELINE_WEIGHTS = None
        print('Multiple best.pt files found. Set BASELINE_OVERRIDE to one of:', baseline_candidates)

print('Dataset input:', DATA_DIR)
print('Baseline checkpoint:', BASELINE_WEIGHTS)
print('Writable outputs:', PROJECT_DIR)


In [ ]:
SOURCE_DATA_ROOT = DATA_DIR
if not all((SOURCE_DATA_ROOT / split / 'images').is_dir() and (SOURCE_DATA_ROOT / split / 'labels').is_dir() for split in ('train', 'val', 'test')):
    raise FileNotFoundError(f'Attached Kaggle dataset must contain train/val/test images and labels folders: {SOURCE_DATA_ROOT}')

source_yamls = list(SOURCE_DATA_ROOT.glob('*.yaml')) + list(SOURCE_DATA_ROOT.glob('*.yml'))
if not source_yamls:
    source_yamls = list(SOURCE_DATA_ROOT.rglob('*.yaml')) + list(SOURCE_DATA_ROOT.rglob('*.yml'))
if not source_yamls:
    raise FileNotFoundError('Dataset data.yaml with class names was not found.')
source_config = yaml.safe_load(source_yamls[0].read_text(encoding='utf-8'))
raw_names = source_config.get('names')
if isinstance(raw_names, dict):
    class_names = [raw_names[i] for i in sorted(raw_names)]
else:
    class_names = list(raw_names or [])
if len(class_names) != 15:
    raise ValueError(f'Expected the project 15-class IDD label map, found {len(class_names)} class names: {class_names}')
if source_config.get('nc', len(class_names)) != len(class_names):
    raise ValueError('Dataset nc does not match the number of class names.')

# Kaggle inputs are read-only. Build a small writable dataset view: symlink the
# original images, copy valid labels, drop only malformed train rows, and exclude
# any val/test image whose ground-truth annotation is malformed.
image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
CLEAN_DATA_ROOT = Path('/kaggle/temp') / ('astra_adas_dataset_clean_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
invalid_rows = []
source_counts = {}
clean_counts = {}
class_counts = Counter()
missing_label_count = {}
excluded_eval_images = {'val': [], 'test': []}

for split in ('train', 'val', 'test'):
    source_images_dir = SOURCE_DATA_ROOT / split / 'images'
    source_labels_dir = SOURCE_DATA_ROOT / split / 'labels'
    clean_images_dir = CLEAN_DATA_ROOT / split / 'images'
    clean_labels_dir = CLEAN_DATA_ROOT / split / 'labels'
    images = [p for p in source_images_dir.rglob('*') if p.is_file() and p.suffix.lower() in image_extensions]
    if not images:
        raise ValueError(f'No images found in {source_images_dir}')
    source_counts[split] = len(images)
    clean_count = 0
    missing = 0

    for image_path in images:
        relative_path = image_path.relative_to(source_images_dir)
        source_label_path = source_labels_dir / relative_path.with_suffix('.txt')
        valid_rows = []
        image_invalid_rows = []
        if source_label_path.is_file():
            for line_number, row in enumerate(source_label_path.read_text(encoding='utf-8').splitlines(), 1):
                if not row.strip():
                    continue
                fields = row.split()
                try:
                    values = [float(value) for value in fields]
                    class_id = int(values[0])
                    valid = (len(values) == 5 and values[0].is_integer() and 0 <= class_id < len(class_names) and all(0.0 <= v <= 1.0 for v in values[1:]) and values[3] > 0.0 and values[4] > 0.0)
                except (ValueError, IndexError):
                    valid = False
                    class_id = -1
                if valid:
                    valid_rows.append(row.strip())
                    if split == 'train':
                        class_counts[class_names[class_id]] += 1
                else:
                    item = {'split': split, 'label_file': str(source_label_path), 'line': line_number, 'row': row}
                    invalid_rows.append(item)
                    image_invalid_rows.append(item)
        else:
            missing += 1

        if image_invalid_rows and split in ('val', 'test'):
            excluded_eval_images[split].append(str(image_path))
            continue

        clean_image_path = clean_images_dir / relative_path
        clean_label_path = clean_labels_dir / relative_path.with_suffix('.txt')
        clean_image_path.parent.mkdir(parents=True, exist_ok=True)
        clean_label_path.parent.mkdir(parents=True, exist_ok=True)
        clean_image_path.symlink_to(image_path.resolve())
        clean_label_path.write_text('\n'.join(valid_rows) + ('\n' if valid_rows else ''), encoding='utf-8')
        clean_count += 1

    clean_counts[split] = clean_count
    missing_label_count[split] = missing

if not clean_counts['val'] or not clean_counts['test']:
    raise ValueError('Cleaning malformed annotations removed every image from validation or test; inspect the audit before training.')

DATA_ROOT = CLEAN_DATA_ROOT
AUDIT_REPORT = EXPORT_DIR / 'dataset_label_audit.json'
AUDIT_REPORT.write_text(json.dumps({
    'source_dataset': str(SOURCE_DATA_ROOT),
    'training_policy': 'Invalid annotation rows are omitted from writable label copies; source Kaggle dataset is unchanged.',
    'evaluation_policy': 'Validation/test images containing any invalid annotation are excluded to avoid evaluating against incomplete ground truth.',
    'source_image_counts': source_counts,
    'clean_image_counts': clean_counts,
    'training_instances_by_class': dict(class_counts),
    'missing_label_files': missing_label_count,
    'excluded_validation_test_images': excluded_eval_images,
    'invalid_rows': invalid_rows,
}, indent=2), encoding='utf-8')

print('Source Kaggle dataset:', SOURCE_DATA_ROOT)
print('Writable cleaned dataset view:', DATA_ROOT)
print('Classes:', len(class_names), class_names)
print('Source images:', source_counts)
print('Images used after audit:', clean_counts)
print('Images without label files (treated as negatives):', missing_label_count)
print('Invalid annotation rows:', len(invalid_rows))
print('Evaluation images excluded due to malformed labels:', {k: len(v) for k, v in excluded_eval_images.items()})
print('Training-label instances by class after dropping invalid rows:')
for name in class_names:
    print(f'  {name}: {class_counts[name]}')
low_sample_classes = {name: count for name, count in class_counts.items() if count < 100}
if low_sample_classes:
    print('WARNING: classes with fewer than 100 training instances are unlikely to reach reliable high recall without more labeled examples:', low_sample_classes)
print('Audit report:', AUDIT_REPORT)
for item in invalid_rows:
    print(f"  {item['split']} {item['label_file']}:{item['line']}: {item['row']}")


In [ ]:
DATA_YAML = PROJECT_DIR / 'idd_kaggle.yaml'
data_config = {
    'path': str(DATA_ROOT),
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'names': class_names,
    'nc': len(class_names),
}
DATA_YAML.write_text(yaml.safe_dump(data_config, sort_keys=False), encoding='utf-8')
print(DATA_YAML.read_text(encoding='utf-8'))


In [ ]:
def normalize_names(names):
    if isinstance(names, dict):
        return [names[i] for i in sorted(names)]
    return list(names)

def evaluate_checkpoint(weights_path, image_sizes=(640, 768, 960), split='val'):
    model = YOLO(str(weights_path))
    model_names = normalize_names(model.names)
    if model_names != class_names:
        raise ValueError(f'Checkpoint class names do not match dataset names. Checkpoint={model_names}; dataset={class_names}')
    measured = {}
    for image_size in image_sizes:
        metrics = model.val(
            data=str(DATA_YAML), split=split, imgsz=image_size, batch=1,
            workers=4, device=0, plots=False, verbose=False,
        )
        box = metrics.box
        speed = metrics.speed
        preprocess_ms = float(speed.get('preprocess', 0.0))
        inference_ms = float(speed.get('inference', 0.0))
        postprocess_ms = float(speed.get('postprocess', 0.0))
        total_ms = preprocess_ms + inference_ms + postprocess_ms
        row = {
            'mAP50': float(box.map50),
            'mAP50_95': float(box.map),
            'validation_batch_size': 1,
            'preprocess_ms_per_image': preprocess_ms,
            'inference_ms_per_image': inference_ms,
            'postprocess_ms_per_image': postprocess_ms,
            'validation_pipeline_fps': 1000.0 / total_ms if total_ms > 0 else None,
            'inference_only_fps': 1000.0 / inference_ms if inference_ms > 0 else None,
        }
        if hasattr(box, 'ap50') and hasattr(box, 'ap_class_index'):
            row['per_class_AP50'] = {
                class_names[int(class_id)]: float(ap)
                for class_id, ap in zip(box.ap_class_index, box.ap50)
            }
        measured[str(image_size)] = row
        print(f"{weights_path.name} | {split} | {image_size}px | mAP50={row['mAP50']:.4f} | mAP50-95={row['mAP50_95']:.4f} | val FPS={row['validation_pipeline_fps']:.1f} (batch=1)")
    return measured

BASELINE_RESULTS = {}
if BASELINE_WEIGHTS is not None and BASELINE_WEIGHTS.is_file():
    BASELINE_RESULTS = evaluate_checkpoint(BASELINE_WEIGHTS)
else:
    print('Baseline comparison skipped. Attach the baseline best.pt as a Kaggle input and set BASELINE_OVERRIDE to its file path for a paired comparison.')


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is available. In Kaggle, open Notebook options > Accelerator > GPU, then rerun the setup cells.')

EPOCHS = 100
IMAGE_SIZE = 960
BATCH_SIZE = -1  # Ultralytics selects a batch size based on this Kaggle GPU's available VRAM.
WORKERS = min(8, os.cpu_count() or 2)
RUN_NAME = 'astra_m5_yolov8s_960_' + datetime.now().strftime('%Y%m%d_%H%M%S')
PRETRAINED_WEIGHTS = 'yolov8s.pt'  # Or set to an attached Kaggle input path, e.g. '/kaggle/input/yolo-weights/yolov8s.pt'.

model = YOLO(PRETRAINED_WEIGHTS)
train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    workers=WORKERS,
    device=0,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=False,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,
    weight_decay=0.0005,
    mosaic=1.0,
    mixup=0.1,
    degrees=3.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    close_mosaic=10,
    patience=25,
    save_period=10,
    plots=True,
    val=True,
    seed=42,
    deterministic=False,
    cache=False,
    amp=True,
    verbose=True,
)

RUN_DIR = Path(train_results.save_dir)
BEST_WEIGHTS = RUN_DIR / 'weights' / 'best.pt'
if not BEST_WEIGHTS.is_file():
    raise FileNotFoundError(f'Training finished without the expected checkpoint: {BEST_WEIGHTS}')
print('Training run:', RUN_DIR)
print('Candidate checkpoint:', BEST_WEIGHTS)
print('Training log:', RUN_DIR / 'results.csv')


In [ ]:
CANDIDATE_RESULTS = evaluate_checkpoint(BEST_WEIGHTS)
print('\nValidation comparison (same full val split):')
for image_size, candidate in CANDIDATE_RESULTS.items():
    baseline = BASELINE_RESULTS.get(image_size)
    if baseline:
        delta = candidate['mAP50'] - baseline['mAP50']
        print(f"  {image_size}px: baseline mAP50={baseline['mAP50']:.4f}, candidate={candidate['mAP50']:.4f}, delta={delta:+.4f}; val FPS baseline={baseline['validation_pipeline_fps']:.1f}, candidate={candidate['validation_pipeline_fps']:.1f}")
    else:
        print(f"  {image_size}px: candidate mAP50={candidate['mAP50']:.4f}, val FPS={candidate['validation_pipeline_fps']:.1f}; no paired baseline available")
    weakest = sorted(candidate.get('per_class_AP50', {}).items(), key=lambda item: item[1])[:5]
    if weakest:
        print('    Lowest candidate class AP50:', ', '.join(f'{name}={score:.3f}' for name, score in weakest))

EXPORT_WEIGHTS = EXPORT_DIR / f'{RUN_NAME}_best.pt'
if EXPORT_WEIGHTS.exists():
    raise FileExistsError(f'Refusing to overwrite an existing export: {EXPORT_WEIGHTS}')
shutil.copy2(BEST_WEIGHTS, EXPORT_WEIGHTS)
REPORT_PATH = EXPORT_DIR / f'{RUN_NAME}_validation.json'
REPORT_PATH.write_text(json.dumps({
    'run_name': RUN_NAME,
    'dataset_split': 'val',
    'candidate_weights': str(EXPORT_WEIGHTS),
    'baseline_weights': str(BASELINE_WEIGHTS) if BASELINE_RESULTS else None,
    'baseline': BASELINE_RESULTS,
    'candidate': CANDIDATE_RESULTS,
}, indent=2), encoding='utf-8')
print('\nCandidate exported to:', EXPORT_WEIGHTS)
print('Validation report:', REPORT_PATH)


In [ ]:
# Change to True only after selecting/finalizing the candidate from validation.
# This evaluates the frozen candidate (and baseline if attached) on test; do not use these results to tune.
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    FINAL_TEST_RESULTS = {'candidate': evaluate_checkpoint(EXPORT_WEIGHTS, image_sizes=(960,), split='test')}
    if BASELINE_WEIGHTS is not None and BASELINE_WEIGHTS.is_file():
        FINAL_TEST_RESULTS['baseline'] = evaluate_checkpoint(BASELINE_WEIGHTS, image_sizes=(960,), split='test')
    else:
        print('No baseline checkpoint attached; running candidate-only final test.')
    test_report = EXPORT_DIR / f'{RUN_NAME}_final_test.json'
    test_report.write_text(json.dumps(FINAL_TEST_RESULTS, indent=2), encoding='utf-8')
    print('Final test report:', test_report)
else:
    print('Final test not run. Set RUN_FINAL_TEST=True only after model selection is frozen.')


## After the run

- Review the training plots, `results.csv`, and validation JSON under `/kaggle/working/astra_adas_training/`. The default is 100 epochs with early stopping; the auto batch size adapts to the hosted GPU. If you hit an out-of-memory error, lower `IMAGE_SIZE` to 768 or set `BATCH_SIZE = 4` and rerun training.
- Compare mAP, per-class AP, and batch-one validation FPS across 640/768/960; no target accuracy or FPS is guaranteed.
- Validation FPS is measured on the Kaggle GPU and excludes video decoding, tracking, lane/traffic processing, drawing, and dashboard I/O. Benchmark the exported model on the target local hardware before claiming dashboard FPS.
- The test cell is disabled by default. After selecting and freezing the candidate from validation, enable it once for final held-out evaluation; do not tune using test results.
- Preserve outputs by selecting **Save Version** and enabling notebook outputs, or download files from `/kaggle/working`. To use a candidate locally, back up `models/detection/best.pt` first and replace it only after reviewing the final benchmark.
- This detector training does not train traffic-light state, traffic-sign types, lane detection, calibrated distance/speed, tracking, recommendations, or feedback-loop components.